# Mixup + CutMix

Per-sample (timm 'elem' mode) Mixup + CutMix for slipstream. Each image in a batch independently rolls:
1. Bernoulli on `prob` — mix or no-op
2. If both `mixup_alpha > 0` and `cutmix_alpha > 0`, Bernoulli on `switch_prob` — mixup vs cutmix
3. λ sampled from the appropriate Beta

Plugged into `SlipstreamLoader` via the new `after_batch_transforms=[Mixup(...)]` argument.

Pairing follows timm: sample `i` mixes with sample `B-i-1` (the reversed batch).

In [ ]:
LITDATA_VAL_PATH = "s3://visionlab-datasets/imagenet1k/pre-processed/s256-l512-jpgbytes-q100-streaming/val/"

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from slipstream import (
    SlipstreamDataset,
    SlipstreamLoader,
    DecodeRandomResizedCrop,
)
from slipstream.transforms import Mixup, Normalize, IMAGENET_MEAN, IMAGENET_STD

dataset = SlipstreamDataset(
    remote_dir=LITDATA_VAL_PATH,
    decode_images=False,
)
print(f"Dataset: {len(dataset):,} samples")

In [ ]:
def load_batch(dataset, size=224, batch_size=8):
    """Load one batch as float-in-[0,1] images plus int64 labels."""
    dec = DecodeRandomResizedCrop(size=size, to_tensor=True, permute=True)
    loader = SlipstreamLoader(
        dataset, batch_size=batch_size, shuffle=False,
        pipelines={'image': [dec]},
        verbose=False,
    )
    batch = next(iter(loader))
    loader.shutdown()
    return {
        'image': batch['image'].float() / 255.0,
        'label': batch['label'].long(),
    }


def show_grid(rows, row_labels=None, col_titles=None, suptitle=None,
              mean=None, std=None, max_cols=8):
    """Show one or more rows of [B, C, H, W] tensors."""
    if isinstance(rows, torch.Tensor):
        rows = [rows]
    n_rows = len(rows)
    n_cols = min(max_cols, rows[0].shape[0])
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.0 * n_cols, 2.2 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]
    for r, batch in enumerate(rows):
        for c in range(n_cols):
            img = batch[c].detach().clone().float()
            if mean is not None and std is not None:
                m = torch.tensor(mean).view(-1, 1, 1)
                s = torch.tensor(std).view(-1, 1, 1)
                img = img * s + m
            img = img.permute(1, 2, 0).clamp(0, 1).cpu().numpy()
            axes[r, c].imshow(img)
            axes[r, c].axis('off')
            if r == 0 and col_titles is not None and c < len(col_titles):
                axes[r, c].set_title(col_titles[c], fontsize=9)
        if row_labels:
            axes[r, 0].set_title(row_labels[r], fontsize=10, loc='left', x=-0.05, y=0.4)
    if suptitle:
        fig.suptitle(suptitle, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


batch = load_batch(dataset, size=224, batch_size=8)
print(f"image: {tuple(batch['image'].shape)}, label: {tuple(batch['label'].shape)}")

## Pure mixup (cutmix_alpha=0)

Every sample mixes via `λ*x + (1-λ)*x.flip(0)`. Each sample gets its own λ.

In [ ]:
m = Mixup(mixup_alpha=0.8, cutmix_alpha=0.0, prob=1.0,
          num_classes=1000, seed=42)
out = m({'image': batch['image'].clone(), 'label': batch['label'].clone()})
lams = m.last_lam.tolist()
show_grid(
    [batch['image'], out['image']],
    row_labels=['original', 'mixup'],
    col_titles=[f'λ={l:.2f}' for l in lams],
    suptitle="Pure mixup — each sample mixed with x.flip(0) at its own λ",
)

## Pure cutmix (mixup_alpha=0)

Each sample's bbox region is replaced with the bbox region of `x.flip(0)`.

In [ ]:
m = Mixup(mixup_alpha=0.0, cutmix_alpha=1.0, prob=1.0,
          num_classes=1000, seed=42)
out = m({'image': batch['image'].clone(), 'label': batch['label'].clone()})
lams = m.last_lam.tolist()
show_grid(
    [batch['image'], out['image']],
    row_labels=['original', 'cutmix'],
    col_titles=[f'λ={l:.2f}' for l in lams],
    suptitle="Pure cutmix — λ shown is the corrected value (1 - bbox_area / total_area)",
)

## Per-sample randomness (the slipstream way)

With both alphas active and `prob<1`, a single batch contains a mix of mixup'd, cutmix'd, and untouched samples. Each cell labeled with its op + λ.

In [ ]:
m = Mixup(mixup_alpha=0.8, cutmix_alpha=1.0, prob=0.7, switch_prob=0.5,
          num_classes=1000, seed=2026)
out = m({'image': batch['image'].clone(), 'label': batch['label'].clone()})

labels = []
for i, lam in enumerate(m.last_lam.tolist()):
    if m.last_use_mixup[i]:
        labels.append(f"mixup\nλ={lam:.2f}")
    elif m.last_use_cutmix[i]:
        labels.append(f"cutmix\nλ={lam:.2f}")
    else:
        labels.append("no-op")

show_grid(
    [batch['image'], out['image']],
    row_labels=['original', 'mixed'],
    col_titles=labels,
    suptitle="Per-sample: each image rolls its own op and λ",
)

## `cutmix_minmax` — explicit bbox ratio bounds

Each sample's bbox h/w is drawn uniformly in `[min*H, max*H]` (and likewise for W), instead of being derived from λ.

In [ ]:
m = Mixup(mixup_alpha=0.0, cutmix_alpha=0.0, cutmix_minmax=(0.3, 0.5),
          prob=1.0, num_classes=1000, seed=11)
out = m({'image': batch['image'].clone(), 'label': batch['label'].clone()})
yl, yh, xl, xh = m.last_bbox
col_titles = [f"h={int(yh[i]-yl[i])}\nw={int(xh[i]-xl[i])}" for i in range(len(yl))]
show_grid(
    [batch['image'], out['image']],
    row_labels=['original', 'cutmix_minmax'],
    col_titles=col_titles,
    suptitle="cutmix_minmax=(0.3, 0.5) — each sample's bbox h/w in [0.3*H, 0.5*H]",
)

## Label smoothing

Without smoothing, mixed labels are sparse two-hot. With `label_smoothing=0.1`, the off-class probability is bumped to `0.1 / num_classes` and the two on-class entries are reduced accordingly.

In [ ]:
K = 20  # small for visualization
small_labels = torch.randint(0, K, (8,))
small_batch = {'image': batch['image'].clone(), 'label': small_labels}

m_smooth = Mixup(mixup_alpha=0.8, cutmix_alpha=0.0, prob=1.0,
                 label_smoothing=0.1, num_classes=K, seed=7)
out_smooth = m_smooth({**small_batch, 'image': batch['image'].clone()})

m_hard = Mixup(mixup_alpha=0.8, cutmix_alpha=0.0, prob=1.0,
               label_smoothing=0.0, num_classes=K, seed=7)
out_hard = m_hard({**small_batch, 'image': batch['image'].clone()})

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].bar(np.arange(K), out_hard['label'][0].numpy())
axes[0].set_title(f"sample 0, smoothing=0.0  (λ={m_hard.last_lam[0]:.2f})")
axes[0].set_ylabel('probability')
axes[1].bar(np.arange(K), out_smooth['label'][0].numpy())
axes[1].set_title(f"sample 0, smoothing=0.1  (λ={m_smooth.last_lam[0]:.2f})")
axes[1].set_xlabel('class index')
axes[1].set_ylabel('probability')
for ax in axes:
    ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()
print(f"original classes: y1={int(small_labels[0])}, y2={int(small_labels.flip(0)[0])}")

## End-to-end via `SlipstreamLoader.after_batch_transforms`

The Mixup transform is plugged into the loader directly — it runs after per-field pipelines, on the assembled batch dict.

In [ ]:
from slipstream.transforms import ToTorchImage, ToFloatDiv

loader = SlipstreamLoader(
    dataset, batch_size=8, shuffle=False,
    pipelines={
        'image': [
            DecodeRandomResizedCrop(size=224, to_tensor=True, permute=True),
            ToFloatDiv(255.0),
        ],
    },
    after_batch_transforms=[
        Mixup(mixup_alpha=0.8, cutmix_alpha=1.0, prob=0.8, switch_prob=0.5,
              label_smoothing=0.1, num_classes=1000, seed=2026),
    ],
    verbose=False,
)
batch_out = next(iter(loader))
loader.shutdown()

print(f"image: {tuple(batch_out['image'].shape)}")
print(f"label: {tuple(batch_out['label'].shape)}  dtype={batch_out['label'].dtype}")
print(f"label sums (should be ~1): {batch_out['label'].sum(dim=-1).tolist()}")
show_grid(batch_out['image'], suptitle="after Mixup via after_batch_transforms")

In [ ]:
batch_out['label'].shape